# Optional 11b: nested-design DE (limma-voom + duplicateCorrelation)
Run after the main audit notebook has finished cell 11 (it writes `pb_counts.csv` and `pb_meta.csv` to `plantag_audit/`). Results go to `plantag_audit/bundle/09b_limma_*.csv` and the bundle is re-zipped. First run installs limma/edgeR (a few minutes).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 11b-1 | Make R available in this Colab session
%load_ext rpy2.ipython

In [ ]:
%%R
# 11b | Optional: nested-design DE with limma-voom + duplicateCorrelation, one fit per time point.
# Leaves are nested in capture areas (sections): leaf-level DESeq2 treats them as independent (anti-conservative),
# section-level DESeq2 has 2 vs 2 (underpowered). duplicateCorrelation estimates how similar leaves of the same
# section are and lmFit uses that correlation, which sits between the two. Inputs: pb_counts.csv, pb_meta.csv (cell 11).
OUT    <- "/content/drive/MyDrive/spatial_transcriptomics/plantag_audit"     # same OUT as in cell 1
BUNDLE <- file.path(OUT, "bundle")
EXCLUDE_REGEX <- "^TYLCV|^CQW56_"     # viral features + mitochondrial genes; add plastid IDs once confirmed, e.g. "|^LyesC2"

for (p in c("limma", "edgeR")) if (!requireNamespace(p, quietly = TRUE)) {            # first run per session: a few minutes
  if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager", repos = "https://cloud.r-project.org")
  BiocManager::install(p, update = FALSE, ask = FALSE)
}
suppressMessages({library(limma); library(edgeR)})

cnt_all  <- t(as.matrix(read.csv(file.path(OUT, "pb_counts.csv"), row.names = 1, check.names = FALSE)))
meta_all <- read.csv(file.path(OUT, "pb_meta.csv"), row.names = 1, check.names = FALSE)
use      <- !grepl("_fragment$|^out_of_tissue$", meta_all$leaf)                      # whole leaves only
cnt_all  <- cnt_all[!grepl(EXCLUDE_REGEX, rownames(cnt_all)), use, drop = FALSE]
meta_all <- meta_all[use, , drop = FALSE]

summ <- list()
for (tp in sort(unique(meta_all$timepoint))) {
  m <- meta_all[meta_all$timepoint == tp, , drop = FALSE]
  m$condition <- factor(m$condition, levels = c("control", "infected"))
  n_pb  <- table(m$condition)
  n_sec <- tapply(m$sample, m$condition, function(x) length(unique(x)))
  if (any(is.na(n_sec) | n_sec < 2 | n_pb < 2)) { cat(tp, ": skipped, needs >= 2 sections and >= 2 leaves per condition\n"); next }
  y <- DGEList(cnt_all[, rownames(m), drop = FALSE])
  y <- calcNormFactors(y[filterByExpr(y, group = m$condition), , keep.lib.sizes = FALSE])
  design <- model.matrix(~ condition, m)
  v   <- voom(y, design)                                                            # round 1
  rho <- duplicateCorrelation(v, design, block = m$sample)$consensus
  v   <- voom(y, design, block = m$sample, correlation = rho)                       # round 2 (standard)
  rho <- duplicateCorrelation(v, design, block = m$sample)$consensus
  rho_used <- if (is.na(rho)) 0 else max(rho, 0)                                    # negative/NA -> leaves treated as independent
  fit <- eBayes(lmFit(v, design, block = m$sample, correlation = rho_used))
  res <- topTable(fit, coef = "conditioninfected", number = Inf, sort.by = "P")
  write.csv(res, file.path(BUNDLE, sprintf("09b_limma_dupcor_%s.csv", tp)))
  summ[[tp]] <- data.frame(timepoint = tp, leaves_control = n_pb[["control"]], leaves_infected = n_pb[["infected"]],
                           sections_control = n_sec[["control"]], sections_infected = n_sec[["infected"]],
                           genes_tested = nrow(res), within_section_correlation = round(rho, 3), correlation_used = round(rho_used, 3),
                           fdr05_up = sum(res$adj.P.Val < 0.05 & res$logFC > 0), fdr05_down = sum(res$adj.P.Val < 0.05 & res$logFC < 0))
  print(summ[[tp]], row.names = FALSE)
}
if (length(summ)) write.csv(do.call(rbind, summ), file.path(BUNDLE, "09b_limma_dupcor_summary.csv"), row.names = FALSE)

In [ ]:
# 11b-3 | Compare limma (nested design) with the DESeq2 two-level results, then re-zip the bundle
import shutil
import pandas as pd
from pathlib import Path
from scipy.stats import spearmanr
BD = Path("/content/drive/MyDrive/spatial_transcriptomics/plantag_audit") / "bundle"
rows = []
for lp in sorted(BD.glob("09b_limma_dupcor_*dpi.csv")):
    tp = lp.stem.split("_")[-1]
    L = pd.read_csv(lp, index_col=0)
    row = {"timepoint": tp, "limma_fdr05": int((L["adj.P.Val"] < 0.05).sum())}
    dp = BD / f"09_DE_{tp}_all_genes_top300.csv"
    if dp.exists():
        D = pd.read_csv(dp, index_col=0)
        robust = D.index[D["robust"].astype(bool)]
        common = D.index.intersection(L.index)
        row.update({"deseq2_robust": len(robust),
                    "deseq2_robust_also_limma_fdr05": int((L.reindex(robust)["adj.P.Val"] < 0.05).sum()),
                    "logFC_spearman_on_deseq2_top300": round(spearmanr(D.loc[common, "log2FoldChange_leaf"], L.loc[common, "logFC"])[0], 3)})
    rows.append(row)
cmp = pd.DataFrame(rows)
display(cmp)
cmp.to_csv(BD / "09b_limma_vs_deseq2.csv", index=False)
print("bundle re-zipped:", shutil.make_archive(str(BD), "zip", BD))